# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [3]:
dfSynonym=pd.read_excel(os.path.join(cwd,'Synonyms_filtered_v3.xlsx'),engine="openpyxl")
dfSynonym=dfSynonym.sort_values(by="Symbol")
dfSynonym=dfSynonym.reset_index()
dfSynonym=dfSynonym.drop(columns="index")
dfSynonym.index=dfSynonym["Symbol"]

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\david\\OneDrive\\Desktop\\Forbeck_Reporter_Downloader_and_Raw_Data\\Synonyms_filtered_v3.xlsx'

In [4]:
gene_terms=str("(\"+_gene_+\"[ti])")
CT1_queryPM=str("")
CT1_queryNIH=str("")

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

# Search 1: Standard Search with cancer and genes

In [ ]:
url_Pubmed_S1='https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=(cancer[ti]+AND+(\"+_gene_+\"[ti]))&retmax=20'
Out_DF_S1=pd.DataFrame(columns=["Gene name","Pubs[title]","Pubs[title/abstract]", "Number of Grants[title/abstract]", "Award Amount[title/abstract]", "Number of Grants[title]", "Award Amount[title]", "Synonyms"])
PM_Tiab_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NCBI PubMed\\Title and abstract data\\'
PM_Ti_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NCBI PubMed\\Title only data\\'
NIH_Ti_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NIH RePORTER\\Title only data\\'
NIH_Tiab_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NIH RePORTER\\Title and abstract data\\'
paramsDefault = {
        "criteria": {
    "advanced_text_search": {"operator": "advanced", "search_field": "projecttitle,abstracttext","search_text": str("cancer AND ")}},
        "include_fields": ["ApplId", "ProjectTitle", "AwardAmount", "DirectCostAmt", "IndirectCostAmt","ProjectStartDate", "ProjectEndDate"],"offset": 0, "limit": 500, }
Out_DF_S1CD=AccessPubMed_and_Reporter_Master_function(Out_DF_S1,url_Pubmed_S1,paramsDefault,PM_Ti_raw,PM_Tiab_raw,NIH_Ti_raw,NIH_Tiab_raw)
print(Out_DF_S1CD)
df=pd.read_csv(os.path.join(cwd_Raw_Data_outputs,"NIH+PM_TP53.csv"), sep=",", header=0)#path for manually downloaded TP53 title/abstract award amount
df=df.iloc[:, :-1]
df["Total"]=df["Total Cost"]+df["Total Cost (Sub Projects)"]
Total_TP53=0
for i in df["Total"]:
    if (i!="" and i!="  "):#NIH stores empty values as spaces for some reason 
        Total_TP53=int(i)+Total_TP53
Out_DF_S1CD.index=Out_DF_S1CD["Gene name"]
Out_DF_S1CD=Out_DF_S1CD.drop(columns="Gene name")
Out_DF_S1CD.loc["TP53"]["Award Amount[title/abstract]"]=int(Total_TP53)
Out_DF_S1CD.to_excel(os.path.join(cwd_Output,"NIH+PM_Data.xlsx"), engine="openpyxl")

ABI1
title block
{'Gene name': 'ABI1', 'Pubs[title]': 6, 'Pubs[title/abstract]': 35, 'Number of Grants[title/abstract]': 15, 'Award Amount[title/abstract]': 3781139, 'Number of Grants[title]': 0, 'Award Amount[title]': 0, 'Synonyms': 'ABI-1|ABLBP4|E3B1|NAP1BP|SSH3BP|SSH3BP1|'}
ABL1
title block
{'Gene name': 'ABL1', 'Pubs[title]': 114, 'Pubs[title/abstract]': 2258, 'Number of Grants[title/abstract]': 1060, 'Award Amount[title/abstract]': 383784914, 'Number of Grants[title]': 17, 'Award Amount[title]': 2089563, 'Synonyms': 'ABL|BCR-ABL|CHDSKM|JTK7|bcr/abl|c-ABL|c-ABL1|p150|v-abl|'}
ABL2
excluded:  ARG
title block
{'Gene name': 'ABL2', 'Pubs[title]': 6, 'Pubs[title/abstract]': 61, 'Number of Grants[title/abstract]': 22, 'Award Amount[title/abstract]': 2335940, 'Number of Grants[title]': 2, 'Award Amount[title]': 76296, 'Synonyms': 'ABLL|'}
ACKR3
title block
{'Gene name': 'ACKR3', 'Pubs[title]': 6, 'Pubs[title/abstract]': 47, 'Number of Grants[title/abstract]': 16, 'Award Amount[title/abst

In [6]:

xlsx_path = Path("ShortMeeting history.xlsx")  # <-- change if needed

# Read sheets (your first sheet name is a bit odd, so grab by index)
xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.tail(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
54,"Elsa Flores, PhD",2005 Scholar Retreat,Scholar,Elsa,Flores,PhD,MD Anderson Cancer Center,NaN
55,"Kimryn Rathmell, MD, PhD",2005 Scholar Retreat,Scholar,Kimryn,Rathmell,"MD, PhD",Vanderbilt University Medical Center,NaN
56,"Masashi Narita, MD, PhD",2005 Scholar Retreat,Scholar,Masashi,Narita,"MD, PhD",Cambridge Institute,NaN
57,"Jan Karlseder, PhD",2005 Scholar Retreat,Scholar,Jan,Karlseder,PhD,Salk Institute,NaN
58,"James Amatruda, MD, PhD",2005 Scholar Retreat,Scholar,James,Amatruda,"MD, PhD",Memorial Sloan Kettering Cancer Center,NaN


In [7]:
# Cell 2 — helpers (normalize meeting titles + parse chair names)

def normalize_meeting_title(x: str) -> str:
    """
    Make meeting titles comparable across sheets:
    - cast to str
    - strip leading/trailing whitespace
    - remove surrounding quotes
    - collapse internal whitespace (including newlines)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    # remove one pair of surrounding quotes if present
    if (len(s) >= 2) and ((s[0] == s[-1]) and s[0] in {"'", '"'}):
        s = s[1:-1].strip()
    s = re.sub(r"\s+", " ", s)  # collapse newlines/tabs/multiple spaces
    return s

def split_chair_names(chairs_cell) -> list[str]:
    """
    Turn the 'Meeting Chairs' cell into a list of chair name strings.
    Handles separators like ';', ',', ' and ', '&', and common ' of ' patterns.
    Keeps credentials as part of the name string (e.g., 'MD, PhD').
    """
    if pd.isna(chairs_cell):
        return []
    s = str(chairs_cell).strip()
    s = re.sub(r"\s+", " ", s)

    # Many entries look like "Name of Institution; Name of Institution"
    # Split primarily on ';' first.
    parts = [p.strip() for p in s.split(";") if p.strip()]

    # Further split each part on " and " / " & " if it contains multiple chairs.
    chairs = []
    for p in parts:
        sub = re.split(r"\s+(?:and|&)\s+", p)
        for item in sub:
            item = item.strip()
            if not item:
                continue
            # Remove trailing institution phrase like " of XYZ" (optional, but helps matching)
            item = re.sub(r"\s+of\s+.+$", "", item).strip()
            chairs.append(item)

    # de-dup while preserving order
    seen = set()
    out = []
    for c in chairs:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

In [8]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Map normalized meeting topic -> year (if duplicates exist, keep the first non-null year)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year onto participants using normalized title
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

participants_df[["Participant", "Meeting", "Year"]]

,Participant,Meeting,Year
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,2023
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,2023
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,2023
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,2023
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,2023
5,"Prof Banafshe Larijani , PhD",Targeting Lipid Biology in Cancer,2023
6,"Ray Blind,",Targeting Lipid Biology in Cancer,2023
7,"Brooke Emerling, PhD",Targeting Lipid Biology in Cancer,2023
8,"Gretchen Alicea, PhD",Targeting Lipid Biology in Cancer,2023
9,"Sarah Skuli,",Targeting Lipid Biology in Cancer,2023


In [9]:
# Cell 4 — (1) create dict keyed by "Meeting Topic (Year)" with list of participant full names

# Keep only rows that have a meeting + participant name
p = participants_df.dropna(subset=["Meeting_norm", "Participant"]).copy()

# Build a key string like "Targeting Lipid Biology in Cancer (2021)"
def make_meeting_year_key(meeting_norm, year):
    y = "" if pd.isna(year) else str(int(year)) if float(year).is_integer() else str(year)
    return f"{meeting_norm} ({y})" if y else f"{meeting_norm} (Year Unknown)"

p["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(p["Meeting_norm"], p["Year"])]

meeting_attendees_dict = (
    p.groupby("MeetingYearKey")["Participant"]
     .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
     .to_dict()
)

# Example: show first 5 keys
list(meeting_attendees_dict["Targeting Lipid Biology in Cancer (2023)"])

['Alison Ringel, PhD',
 'Bart Vanhaesebroeck, PhD',
 'Brooke Emerling, PhD',
 'Christina Mitchell, MB BS, PhD',
 'David Fruman, PhD',
 'Emilio Hirsch, PhD',
 'Gretchen Alicea, PhD',
 'Hua Eleanor Yu, PhD',
 'Jeremy Baskin, PhD',
 'Karen Dixon,',
 'Livia  Schiavinato Eberlin, PhD',
 'Neil Vasan, MD, PhD',
 'Prof Banafshe  Larijani , PhD',
 'Ray Blind,',
 'Sarah  Skuli,',
 'Tamas Balla, MD, PhD',
 'Targeting Lipid Biology in Cancer',
 'Vytas Bankaitis, PhD']

In [ ]:
REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"
_SUFFIXES = {"jr","sr","ii","iii","iv","md","phd","mph","ms","m.d.","ph.d.","dr"}

#compact name parsing (good enough for "First Last" and "Last, First")

def parse_first_last(name):
    if not name: return "", ""
    s = re.sub(r"\([^)]*\)", "", str(name))                              # drop parentheticals
    s = re.sub(r"^(dr\.?|prof\.?)\s+", "", s.strip(), flags=re.I)       # drop honorifics
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if len(parts) >= 2:                                                 # "Last, First ..."
        last = parts[0]
        first = parts[1].split()[0] if parts[1] else ""
        return first.lower(), last.lower()
    toks = [t for t in s.split() if t.lower().strip(".") not in _SUFFIXES]
    if len(toks) == 1: return "", toks[0].lower()
    return toks[0].lower(), toks[-1].lower()

def pi_entry(name):
    first, last = parse_first_last(name)
    if not last: return None
    return {"any_name": last, "first_name": first}  # last name in any_name, optional first_name

#meeting year + bins
def meeting_year_from_key(meeting_key):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def five_year_bins(center_year, n_bins_before=1, n_bins_after=3):
    # e.g. center=2000 -> [(1995,2000),(2000,2005),(2005,2010),(2010,2015)]
    return [(center_year-5*i, center_year-5*(i-1)) for i in range(n_bins_before, 0, -1)] + \
           [(center_year+5*i, center_year+5*(i+1)) for i in range(0, n_bins_after)]

def fiscal_years_for_bin(start, end):
    # inclusive of start..end-1 if you interpret bins as [start,end); but user examples look inclusive.
    # We'll do inclusive for both ends.
    return list(range(int(start), int(end) + 1))

# RePORTER paging + lightweight award extraction
def reporter_search_all_pages(payload, sleep=0.33, limit=500, freeze=None):
    params = deepcopy(payload); params.update({"offset": 0, "limit": limit})
    if freeze and Path(freeze).exists():
        pages = [json.loads(l) for l in Path(freeze).read_text().splitlines()
                 if l and not l.startswith("Date accessed:")]
        total = pages[0].get("meta", {}).get("total", 0) if pages else 0
        return {"total": int(total or 0), "pages": pages}

    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json(); pages.append(page)
        total = total or page.get("meta", {}).get("total", 0)
        off = page.get("meta", {}).get("offset", params["offset"])
        cnt = page.get("meta", {}).get("count", len(page.get("results", [])))
        if cnt == 0 or off + cnt >= total: break
        params["offset"] = off + cnt; time.sleep(sleep)

    if freeze:
        fp = Path(freeze); fp.parent.mkdir(parents=True, exist_ok=True)
        with fp.open("w", encoding="utf-8") as f:
            for p in pages: f.write(json.dumps(p) + "\n")
            f.write(f"Date accessed: {datetime.now().isoformat()}\n")
    return {"total": int(total or 0), "pages": pages}

def sum_awards(pages):
    return sum(int(proj.get("fy_total_cost") or proj.get("award_amount") or 0)
               for page in pages for proj in page.get("results", []))

def flatten_awards(pages, meeting_key, bin_start, bin_end, pair_key):
    rows = []
    for page in pages:
        for proj in page.get("results", []):
            appl = proj.get("appl_id") or proj.get("applId") or proj.get("ApplId")
            subp = proj.get("subproject_id") or proj.get("subprojectId") or proj.get("SubprojectId")
            fy   = proj.get("fiscal_year") or proj.get("fiscalYear") or proj.get("FiscalYear")
            pnum = proj.get("project_num") or proj.get("projectNum") or proj.get("ProjectNum")
            title = proj.get("project_title") or proj.get("projectTitle") or proj.get("ProjectTitle")
            cost = proj.get("fy_total_cost") if proj.get("fy_total_cost") is not None else proj.get("award_amount")
            # PI names often appear under "principal_investigators" depending on include_fields;
            # keep raw for now so you can inspect.
            pis = proj.get("principal_investigators") or proj.get("PrincipalInvestigators")

            rows.append({
                "MeetingYearKey": meeting_key,
                "Bin": f"{bin_start}-{bin_end}",
                "Pair": pair_key,
                "ApplId": appl,
                "SubprojectId": subp,
                "FiscalYear": fy,
                "ProjectNum": pnum,
                "ProjectTitle": title,
                "AwardAmount": int(cost or 0),
                "PrincipalInvestigators_raw": pis
            })
    return rows

# --- main: multi-PI collaborations (>=2 attendees) via PI pairs ---

def multi_pi_collab_by_meeting(
    meeting_attendees_dict,
    NIH_param_template,
    n_bins_before=1,
    n_bins_after=3,
    max_pairs_per_meeting=None,      # set to e.g. 3000 if meetings are huge
    sleep=0.25,
    raw_dir=None
):
    raw_dir = Path(raw_dir) if raw_dir else None
    summary_rows, detail_rows = [], []

    for meeting_key, names in meeting_attendees_dict.items():
        print("testing main function")
        year = meeting_year_from_key(meeting_key)
        if year is None:
            summary_rows.append({"MeetingYearKey": meeting_key, "Bin": None, "Note": "No year in meeting key"})
            continue
        print(f"year:{year}")
        # build clean PI list (drop unparseable) and keep original for labeling pairs
        clean = []
        for n in names:
            e = pi_entry(n)
            if e: clean.append((n, e))
        print(f"names:{clean}")
        if len(clean) < 2:
            summary_rows.append({"MeetingYearKey": meeting_key, "Bin": None, "Note": "Fewer than 2 parsable names"})
            continue

        # all attendee pairs => collaboration candidates
        pairs = list(itertools.combinations(clean, 2))
        if max_pairs_per_meeting and len(pairs) > max_pairs_per_meeting:
            pairs = pairs[:max_pairs_per_meeting]  # simple cap; swap to smarter sampling if needed

        for (start, end) in five_year_bins(year, n_bins_before, n_bins_after):
            fys = fiscal_years_for_bin(start, end)
            bin_awards = {}      # award_id -> aggregated award record (de-dupe across pairs)
            bin_pair_hits = 0    # how many PI pairs returned >=1 award (not de-duped)

            for (name_a, pi_a), (name_b, pi_b) in pairs:
                payload = deepcopy(NIH_param_template)
                payload.setdefault("criteria", {})
                c = payload["criteria"]
                c["pi_names"] = [pi_a, pi_b]
                c["multi_pi_only"] = True
                c["fiscal_years"] = fys
                c.pop("advanced_text_search", None)  # PI-only
                # reduce response size; keep what you need (adjust as desired)
                payload["include_fields"] = [
                    "appl_id","subproject_id","fiscal_year","project_num","project_title",
                    "fy_total_cost","award_amount","principal_investigators"
                ]

                freeze = None
                if raw_dir:
                    safe_m = re.sub(r"[^A-Za-z0-9._-]+", "_", meeting_key)[:120]
                    safe_p = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{name_a}__{name_b}")[:120]
                    freeze = raw_dir / safe_m / f"{start}_{end}" / f"{safe_p}.jsonl"

                res = reporter_search_all_pages(payload, sleep=sleep, freeze=str(freeze) if freeze else None)
                if res["total"] <= 0:
                    continue

                bin_pair_hits += 1
                pair_key = f"{name_a} || {name_b}"

                # detail rows (pair-level)
                detail_rows += flatten_awards(res["pages"], meeting_key, start, end, pair_key)

                # meeting/bin de-dupe at award level
                for page in res["pages"]:
                    for proj in page.get("results", []):
                        appl = proj.get("appl_id") or proj.get("applId") or proj.get("ApplId")
                        subp = proj.get("subproject_id") or proj.get("subprojectId") or proj.get("SubprojectId")
                        award_id = f"{appl}:{subp}" if subp else str(appl)
                        cost = proj.get("fy_total_cost") if proj.get("fy_total_cost") is not None else proj.get("award_amount")
                        if award_id not in bin_awards:
                            bin_awards[award_id] = int(cost or 0)
                        else:
                            # keep max cost seen for that award_id (avoids inflating if repeated across pair queries)
                            bin_awards[award_id] = max(bin_awards[award_id], int(cost or 0))

                time.sleep(sleep)

            summary_rows.append({
                "MeetingYearKey": meeting_key,
                "MeetingYear": year,
                "Bin": f"{start}-{end}",
                "FiscalYears": f"{fys[0]}..{fys[-1]}",
                "Attendees_parsed": len(clean),
                "Pairs_tested": len(pairs),
                "Pairs_with_hits": bin_pair_hits,
                "Unique_awards": len(bin_awards),
                "Unique_award_total_amount": int(sum(bin_awards.values()))
            })
    print("testing")
    print(detail_rows)
    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)

In [8]:
NIH_param={"criteria": {}}
summary_df, details_df = multi_pi_collab_by_meeting(
    meeting_attendees_dict,
    NIH_param_template=NIH_param,   # your template dict
    n_bins_before=1,                # 1995-2000 for a 2000 meeting
    n_bins_after=3,                 # 2000-2005, 2005-2010, 2010-2015
    max_pairs_per_meeting=None,     # set a cap if meetings have lots of attendees
    sleep=0.25,
    raw_dir="raw_reporter_multi_pi_pairs"  # or None
)

summary_df.sort_values(["MeetingYearKey","Bin"]).head(20)

KeyboardInterrupt: 